# 🔬 SPECTER vs Sentence-Transformers: Scientific Paper Embedding Comparison

This notebook compares two approaches for generating embeddings from scientific papers:

1. **SPECTER**: Document-level representation learning using citation-informed transformers (specifically for scientific papers)
2. **Sentence-Transformers**: General-purpose semantic embeddings (paraphrase-albert-small-v2)

Let's see how domain-specific models perform versus general-purpose ones!

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import torch
import time
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

print("📚 Libraries imported successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")

📚 Libraries imported successfully!
🔥 PyTorch version: 2.2.2


In [2]:
# Load our arXiv dataset
import glob

csv_files = glob.glob('arxiv_papers_*.csv')
if csv_files:
    latest_csv = max(csv_files)
    papers_df = pd.read_csv(latest_csv)
    print(f"📊 Loaded {len(papers_df)} papers from: {latest_csv}")
    
    # Take a representative sample for comparison (to speed up processing)
    sample_size = 200
    papers_sample = papers_df.sample(n=min(sample_size, len(papers_df)), random_state=42)
    papers_sample['combined_text'] = papers_sample['title'] + ' ' + papers_sample['abstract']
    
    print(f"🔬 Working with {len(papers_sample)} papers for comparison")
    
    # Show some sample papers
    print("\n📋 Sample papers:")
    for i, paper in papers_sample.head(3).iterrows():
        print(f"  • {paper['title'][:80]}... ({str(paper['published'])[:4]})")
        
else:
    print("⚠️ No dataset found. Please run the main notebook first.")

📊 Loaded 1000 papers from: arxiv_papers_20250817_080201.csv
🔬 Working with 200 papers for comparison

📋 Sample papers:
  • The Fairness Field Guide: Perspectives from Social and Formal Sciences... (2022)
  • Gender Bias in Machine Translation and The Era of Large Language Models... (2024)
  • Machine Learning and Data Analysis Using Posets: A Survey... (2024)


In [3]:
# Load both models
print("🤖 Loading SPECTER model...")
specter_tokenizer = AutoTokenizer.from_pretrained('allenai/specter')
specter_model = AutoModel.from_pretrained('allenai/specter', use_safetensors=True)
print("✅ SPECTER loaded!")

print("\n🤖 Loading Sentence-Transformer model...")
sentence_model = SentenceTransformer('paraphrase-albert-small-v2')
print("✅ Sentence-Transformer loaded!")

print(f"\n📐 Model dimensions:")
print(f"  SPECTER: 768 dimensions")
print(f"  Sentence-Transformer: {sentence_model.get_sentence_embedding_dimension()} dimensions")

🤖 Loading SPECTER model...


✅ SPECTER loaded!

🤖 Loading Sentence-Transformer model...


✅ Sentence-Transformer loaded!

📐 Model dimensions:
  SPECTER: 768 dimensions
  Sentence-Transformer: 768 dimensions


In [4]:
def generate_specter_embeddings(papers_data):
    """Generate embeddings using SPECTER model"""
    # Format as title [SEP] abstract
    formatted_papers = []
    for _, paper in papers_data.iterrows():
        text = paper['title'] + specter_tokenizer.sep_token + str(paper['abstract'])
        formatted_papers.append(text)
    
    # Tokenize
    inputs = specter_tokenizer(
        formatted_papers, 
        padding=True, 
        truncation=True, 
        return_tensors='pt', 
        max_length=512
    )
    
    # Generate embeddings
    with torch.no_grad():
        result = specter_model(**inputs)
        embeddings = result.last_hidden_state[:, 0, :].numpy()  # [CLS] token
    
    return embeddings

def generate_sentence_embeddings(papers_data):
    """Generate embeddings using Sentence-Transformer model"""
    combined_texts = papers_data['combined_text'].tolist()
    embeddings = sentence_model.encode(combined_texts, show_progress_bar=True)
    return embeddings

print("🛠️ Embedding functions defined!")

🛠️ Embedding functions defined!


In [5]:
# Generate embeddings with both models
print("⏱️ Generating SPECTER embeddings...")
start_time = time.time()
specter_embeddings = generate_specter_embeddings(papers_sample)
specter_time = time.time() - start_time
print(f"✅ SPECTER embeddings generated in {specter_time:.2f} seconds")
print(f"   Shape: {specter_embeddings.shape}")

print("\n⏱️ Generating Sentence-Transformer embeddings...")
start_time = time.time()
sentence_embeddings = generate_sentence_embeddings(papers_sample)
sentence_time = time.time() - start_time
print(f"✅ Sentence-Transformer embeddings generated in {sentence_time:.2f} seconds")
print(f"   Shape: {sentence_embeddings.shape}")

print(f"\n⚡ Speed comparison:")
print(f"   SPECTER: {len(papers_sample)/specter_time:.1f} papers/second")
print(f"   Sentence-Transformer: {len(papers_sample)/sentence_time:.1f} papers/second")

⏱️ Generating SPECTER embeddings...


✅ SPECTER embeddings generated in 103.37 seconds
   Shape: (200, 768)

⏱️ Generating Sentence-Transformer embeddings...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Sentence-Transformer embeddings generated in 10.89 seconds
   Shape: (200, 768)

⚡ Speed comparison:
   SPECTER: 1.9 papers/second
   Sentence-Transformer: 18.4 papers/second


In [6]:
# Define search function for both models
def semantic_search_comparison(query, top_k=5):
    """Compare search results from both models"""
    results = {}
    
    # SPECTER search
    specter_query = query + specter_tokenizer.sep_token + ""  # Add SEP token
    specter_query_inputs = specter_tokenizer(
        [specter_query], 
        padding=True, 
        truncation=True, 
        return_tensors='pt', 
        max_length=512
    )
    
    with torch.no_grad():
        specter_query_result = specter_model(**specter_query_inputs)
        specter_query_embedding = specter_query_result.last_hidden_state[:, 0, :].numpy()
    
    specter_similarities = cosine_similarity(specter_query_embedding, specter_embeddings)[0]
    specter_top_indices = np.argsort(specter_similarities)[::-1][:top_k]
    
    # Sentence-Transformer search
    sentence_query_embedding = sentence_model.encode([query])
    sentence_similarities = cosine_similarity(sentence_query_embedding, sentence_embeddings)[0]
    sentence_top_indices = np.argsort(sentence_similarities)[::-1][:top_k]
    
    # Compile results
    results['specter'] = []
    results['sentence'] = []
    
    for idx in specter_top_indices:
        paper = papers_sample.iloc[idx]
        results['specter'].append({
            'title': paper['title'],
            'similarity': specter_similarities[idx],
            'published': str(paper['published'])[:10],
            'abstract': paper['abstract'][:150] + '...' if len(str(paper['abstract'])) > 150 else paper['abstract']
        })
    
    for idx in sentence_top_indices:
        paper = papers_sample.iloc[idx]
        results['sentence'].append({
            'title': paper['title'],
            'similarity': sentence_similarities[idx],
            'published': str(paper['published'])[:10],
            'abstract': paper['abstract'][:150] + '...' if len(str(paper['abstract'])) > 150 else paper['abstract']
        })
    
    return results

print("🔍 Search comparison function ready!")

🔍 Search comparison function ready!


In [7]:
# Test searches on different queries
test_queries = [
    "machine unlearning",
    "neural network optimization",
    "reinforcement learning algorithms",
    "computer vision deep learning"
]

print("🔬 COMPARATIVE SEARCH RESULTS")
print("=" * 60)

for query in test_queries:
    print(f"\n\n🔍 Query: '{query}'")
    print("-" * 50)
    
    results = semantic_search_comparison(query, top_k=3)
    
    print("\n🔬 SPECTER Results:")
    for i, paper in enumerate(results['specter'], 1):
        print(f"  {i}. {paper['similarity']:.3f} - {paper['title']}")
        print(f"     Published: {paper['published']}")
    
    print("\n🤖 Sentence-Transformer Results:")
    for i, paper in enumerate(results['sentence'], 1):
        print(f"  {i}. {paper['similarity']:.3f} - {paper['title']}")
        print(f"     Published: {paper['published']}")
    
    # Calculate average similarity scores
    specter_avg = np.mean([p['similarity'] for p in results['specter']])
    sentence_avg = np.mean([p['similarity'] for p in results['sentence']])
    
    print(f"\n📊 Average Similarity Scores:")
    print(f"   SPECTER: {specter_avg:.3f}")
    print(f"   Sentence-Transformer: {sentence_avg:.3f}")
    
    if specter_avg > sentence_avg:
        print(f"   🏆 SPECTER wins by {specter_avg - sentence_avg:.3f}")
    else:
        print(f"   🏆 Sentence-Transformer wins by {sentence_avg - specter_avg:.3f}")

🔬 COMPARATIVE SEARCH RESULTS


🔍 Query: 'machine unlearning'
--------------------------------------------------

🔬 SPECTER Results:
  1. 0.888 - Proceedings of the 29th International Conference on Machine Learning
  (ICML-12)
     Published: 2012-07-19
  2. 0.883 - mlpy: Machine Learning Python
     Published: 2012-02-29
  3. 0.871 - Considerations upon the Machine Learning Technologies
     Published: 2009-04-23

🤖 Sentence-Transformer Results:
  1. 0.702 - Towards Machine Unlearning Benchmarks: Forgetting the Personal
  Identities in Facial Recognition Systems
     Published: 2023-11-03
  2. 0.580 - Unmasking Clever Hans Predictors and Assessing What Machines Really
  Learn
     Published: 2019-02-26
  3. 0.541 - How to avoid machine learning pitfalls: a guide for academic researchers
     Published: 2021-08-05

📊 Average Similarity Scores:
   SPECTER: 0.881
   Sentence-Transformer: 0.607
   🏆 SPECTER wins by 0.273


🔍 Query: 'neural network optimization'
----------------------------


🔬 SPECTER Results:
  1. 0.809 - Considerations upon the Machine Learning Technologies
     Published: 2009-04-23
  2. 0.803 - How to avoid machine learning pitfalls: a guide for academic researchers
     Published: 2021-08-05
  3. 0.796 - Proceedings of the 29th International Conference on Machine Learning
  (ICML-12)
     Published: 2012-07-19

🤖 Sentence-Transformer Results:
  1. 0.634 - Multimodal Machine Translation with Reinforcement Learning
     Published: 2018-05-07
  2. 0.564 - Numeric Reward Machines
     Published: 2024-04-30
  3. 0.520 - Private Machine Learning via Randomised Response
     Published: 2020-01-14

📊 Average Similarity Scores:
   SPECTER: 0.803
   Sentence-Transformer: 0.573
   🏆 SPECTER wins by 0.230


🔍 Query: 'computer vision deep learning'
--------------------------------------------------

🔬 SPECTER Results:
  1. 0.805 - Deep Learning and Its Applications to Machine Health Monitoring: A
  Survey
     Published: 2016-12-16
  2. 0.803 - Proceedings of the

In [8]:
# Overlap analysis - how many results are the same?
print("\n🔄 RESULT OVERLAP ANALYSIS")
print("=" * 40)

for query in test_queries:
    results = semantic_search_comparison(query, top_k=5)
    
    specter_titles = set([p['title'] for p in results['specter']])
    sentence_titles = set([p['title'] for p in results['sentence']])
    
    overlap = len(specter_titles.intersection(sentence_titles))
    overlap_percent = (overlap / 5) * 100
    
    print(f"'{query[:25]}...': {overlap}/5 papers overlap ({overlap_percent:.0f}%)")
    
    if overlap < 5:
        print(f"  Unique to SPECTER: {len(specter_titles - sentence_titles)} papers")
        print(f"  Unique to Sentence-T: {len(sentence_titles - specter_titles)} papers")


🔄 RESULT OVERLAP ANALYSIS
'machine unlearning...': 1/5 papers overlap (20%)
  Unique to SPECTER: 4 papers
  Unique to Sentence-T: 4 papers


'neural network optimizati...': 1/5 papers overlap (20%)
  Unique to SPECTER: 4 papers
  Unique to Sentence-T: 4 papers
'reinforcement learning al...': 1/5 papers overlap (20%)
  Unique to SPECTER: 4 papers
  Unique to Sentence-T: 4 papers


'computer vision deep lear...': 0/5 papers overlap (0%)
  Unique to SPECTER: 5 papers
  Unique to Sentence-T: 5 papers


In [9]:
# Detailed comparison for machine unlearning query
print("\n🔄 DETAILED ANALYSIS: Machine Unlearning Query")
print("=" * 60)

query = "machine unlearning"
results = semantic_search_comparison(query, top_k=5)

print("\n🔬 SPECTER Results (Detailed):")
for i, paper in enumerate(results['specter'], 1):
    print(f"\n{i}. {paper['title']}")
    print(f"   Similarity: {paper['similarity']:.3f}")
    print(f"   Published: {paper['published']}")
    print(f"   Abstract: {paper['abstract']}")

print("\n" + "="*60)
print("\n🤖 Sentence-Transformer Results (Detailed):")
for i, paper in enumerate(results['sentence'], 1):
    print(f"\n{i}. {paper['title']}")
    print(f"   Similarity: {paper['similarity']:.3f}")
    print(f"   Published: {paper['published']}")
    print(f"   Abstract: {paper['abstract']}")


🔄 DETAILED ANALYSIS: Machine Unlearning Query



🔬 SPECTER Results (Detailed):

1. Proceedings of the 29th International Conference on Machine Learning
  (ICML-12)
   Similarity: 0.888
   Published: 2012-07-19
   Abstract:   This is an index to the papers that appear in the Proceedings of the 29th
International Conference on Machine Learning (ICML-12). The conference was...

2. mlpy: Machine Learning Python
   Similarity: 0.883
   Published: 2012-02-29
   Abstract:   mlpy is a Python Open Source Machine Learning library built on top of
NumPy/SciPy and the GNU Scientific Libraries. mlpy provides a wide range of
st...

3. Considerations upon the Machine Learning Technologies
   Similarity: 0.871
   Published: 2009-04-23
   Abstract:   Artificial intelligence offers superior techniques and methods by which
problems from diverse domains may find an optimal solution. The Machine
Lear...

4. Benchmark and Survey of Automated Machine Learning Frameworks
   Similarity: 0.863
   Published: 2019-04-26
   Abstract:   Machine learning (ML) has 

In [10]:
# Summary and conclusions
print("\n🎯 ANALYSIS SUMMARY")
print("=" * 40)

# Calculate overall performance metrics
all_results = {}
specter_scores = []
sentence_scores = []

for query in test_queries:
    results = semantic_search_comparison(query, top_k=5)
    specter_avg = np.mean([p['similarity'] for p in results['specter']])
    sentence_avg = np.mean([p['similarity'] for p in results['sentence']])
    specter_scores.append(specter_avg)
    sentence_scores.append(sentence_avg)

print(f"\n📊 Overall Performance:")
print(f"   SPECTER average similarity: {np.mean(specter_scores):.3f} (±{np.std(specter_scores):.3f})")
print(f"   Sentence-T average similarity: {np.mean(sentence_scores):.3f} (±{np.std(sentence_scores):.3f})")

specter_wins = sum(s > t for s, t in zip(specter_scores, sentence_scores))
print(f"\n🏆 Query Wins:")
print(f"   SPECTER: {specter_wins}/{len(test_queries)} queries")
print(f"   Sentence-Transformer: {len(test_queries) - specter_wins}/{len(test_queries)} queries")

print(f"\n⚡ Performance:")
print(f"   SPECTER: {len(papers_sample)/specter_time:.1f} papers/second")
print(f"   Sentence-Transformer: {len(papers_sample)/sentence_time:.1f} papers/second")

print(f"\n🎯 Key Insights:")
if np.mean(specter_scores) > np.mean(sentence_scores):
    print(f"   • SPECTER shows higher average similarity scores for scientific papers")
    print(f"   • Domain-specific training appears to improve relevance")
else:
    print(f"   • Sentence-Transformer shows competitive performance")
    
if specter_time < sentence_time:
    print(f"   • SPECTER is faster at generating embeddings")
else:
    print(f"   • Sentence-Transformer is faster at generating embeddings")
    
print(f"   • Both models show different paper selections, suggesting complementary strengths")
print(f"   • SPECTER may be better for citation-related and academic-specific queries")
print(f"   • Sentence-Transformer may be more versatile for general semantic similarity")


🎯 ANALYSIS SUMMARY



📊 Overall Performance:
   SPECTER average similarity: 0.823 (±0.031)
   Sentence-T average similarity: 0.541 (±0.027)

🏆 Query Wins:
   SPECTER: 4/4 queries
   Sentence-Transformer: 0/4 queries

⚡ Performance:
   SPECTER: 1.9 papers/second
   Sentence-Transformer: 18.4 papers/second

🎯 Key Insights:
   • SPECTER shows higher average similarity scores for scientific papers
   • Domain-specific training appears to improve relevance
   • Sentence-Transformer is faster at generating embeddings
   • Both models show different paper selections, suggesting complementary strengths
   • SPECTER may be better for citation-related and academic-specific queries
   • Sentence-Transformer may be more versatile for general semantic similarity
